# Texture from the grey level co-occurrence matrix (Python)

**Fullstack Remote Sensing** by Rifky Nauval Hendrawan · [Chapter: Spatial Texture Extraction](https://rifkynhsp-byte.github.io/Fullstack-Remote-Sensing/en/14-texture.html) · [Bab (Bahasa Indonesia)](https://rifkynhsp-byte.github.io/Fullstack-Remote-Sensing/id/14-texture.html) · [Open the JavaScript in the Code Editor](https://code.earthengine.google.com/?scriptPath=users/rifkynauvalhsp/Fullstack-Remote-Sensing:ch14_glcm_texture)

Run the cells from top to bottom. The first cell installs what Colab lacks, fetches the book's data and helpers, and signs you in to Earth Engine with your own Google Cloud project ([how to get one](https://developers.google.com/earth-engine/guides/access)).

*Jalankan sel dari atas ke bawah. Sel pertama memasang paket, mengambil data buku, dan masuk ke Earth Engine dengan proyek Google Cloud Anda sendiri.*

In [ ]:
# @title Setup: packages, book data, Earth Engine sign-in
!pip -q install earthengine-api geemap tabulate
!test -d /content/book || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/rifkynhsp-byte/Fullstack-Remote-Sensing.git /content/book
!git -C /content/book sparse-checkout set tools scripts data   # only what the code needs, not the website
import sys, os; sys.path[:0] = ['/content/book/tools', '/content/book/scripts/py']; os.chdir('/content/book')
__file__ = '/content/book/scripts/py/ch14_glcm_texture.py'   # the script finds the book's data relative to itself
PROJECT = "your-cloud-project"  # @param {type:"string"}
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT)
_init = ee.Initialize; ee.Initialize = lambda *a, **k: None   # the script must not sign in again

CHAPTER 14 | Quantise one band, compute GLCM, keep four measures, and test
whether they separate anything. In Python.

A note on the maps below: a thumbnail is computed at its own pixel size. GLCM
over the whole delta at 1,400 pixels wide would be a 40 m texture, not a 10 m
one, so the maps show a 5 km window where the output pixel is near native.

In [ ]:
import ee
import matplotlib.pyplot as plt

from book_labels import CLASS_NAMES, LULC_PALETTE, labelled_points

aoi = ee.Geometry.Rectangle([117.30, -1.05, 117.85, -0.60])
window = ee.Geometry.Rectangle([117.45, -0.78, 117.50, -0.73])   # ponds meet mangrove

In [ ]:
def mask_and_scale(i):
    scl = i.select("SCL")
    return (i.updateMask(scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)))
            .divide(10000).select("B.*"))


composite = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
             .filterDate("2023-01-01", "2023-12-31").filterBounds(aoi)
             .map(mask_and_scale).median().clip(aoi))

QUANT_LEVELS = 255
nir = (composite.select("B8").unitScale(0, 0.5).clamp(0, 1)
       .multiply(QUANT_LEVELS).toByte().rename("B8_quantised"))

WINDOW = 3
glcm = nir.glcmTexture(size=WINDOW)
texture = glcm.select(
    ["B8_quantised_contrast", "B8_quantised_ent", "B8_quantised_asm", "B8_quantised_corr"],
    ["tex_contrast", "tex_entropy", "tex_asm", "tex_correlation"])

# PART 5. Does it separate anything? WorldCover labels until you have your own.
training_points = labelled_points(aoi, per_class=(200, 200, 100, 100, 100))
separability = texture.sampleRegions(collection=training_points, properties=["landcover"],
                                     scale=10, tileScale=4)

In [ ]:
def plot_texture_separability(df):
    """Entropy against contrast, one colour per class."""
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for k, name in enumerate(CLASS_NAMES):
        part = df[df["landcover"] == k]
        ax.scatter(part["tex_entropy"], part["tex_contrast"], s=7, alpha=0.6,
                   color="#" + LULC_PALETTE[k], label=name)
    ax.set_yscale("symlog")
    ax.set_xlabel("Entropy (3 × 3 window)")
    ax.set_ylabel("Contrast (symlog)")
    ax.set_title("Entropy against contrast, by class", loc="left")
    ax.legend(frameon=False, fontsize=7, markerscale=2)
    return fig

In [ ]:
def products():
    src = "Sentinel-2 B8, 2023 median, GLCM 3 × 3. GEE."
    return [
        {"kind": "map", "name": "ch14-window-rgb", "image": composite, "region": window,
         "vis": {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3},
         "title": "True colour, 5 km window", "source": src,
         "caption": "Aquaculture ponds cut into mangrove on the delta. In true colour "
                    "the pond walls and the canopy read as similar greens."},
        {"kind": "map", "name": "ch14-entropy", "image": texture.select("tex_entropy"),
         "region": window, "vis": {"min": 0, "max": 4, "palette": ["000000", "14a37f", "ffffff"]},
         "legend": "GLCM entropy", "title": "Entropy", "source": src,
         "caption": "Entropy: smooth water is black, the broken canopy is bright, and the "
                    "straight pond walls draw themselves."},
        {"kind": "map", "name": "ch14-contrast", "image": texture.select("tex_contrast"),
         "region": window, "vis": {"min": 0, "max": 300, "palette": ["000000", "c8792b", "ffffff"]},
         "legend": "GLCM contrast", "title": "Contrast", "source": src,
         "caption": "Contrast peaks along edges, which is why it helps with narrow "
                    "fringes and hurts inside large uniform stands."},
        {"kind": "chart", "name": "ch14-separability", "data": separability,
         "plot": plot_texture_separability,
         "caption": "Texture measured at WorldCover-labelled points. Worth its cost only "
                    "if it pulls apart classes the spectra could not."},
    ]

### The chapter's maps, charts and tables
The same products the book shows, drawn here. Maps also open as an interactive layer.

In [ ]:
from colab_render import show_all
show_all(products())